# Surgical Instrument Force & Motion Analysis

Analyzes sequence recordings (`*.igs.mha`, IGSIO/PLUS metafile format) of surgical
training trials. Each file contains, per frame:

| Field | Meaning |
|---|---|
| `*TipToWorldTransform` | 4×4 pose of each instrument tip (**Bipolar**, **Cavitron**, **Scissors**) |
| `Force` | `fx fy fz tx ty tz` — 3D force + 3D torque from the force sensor |
| `BipolarCollectedPoint0..3` | 4 fiducial points used to register trials into a common frame |
| `Timestamp` | frame time in seconds |

**What this notebook produces**

1. **Point-wise rigid registration** of every trial onto a common reference frame,
   using the 4 collected fiducials (all reported tracking data share this frame).
2. **Time normalized to [0, 1]** per trial so recordings of different length align.
3. **Force magnitude** `√(fx²+fy²+fz²)` per trial.
4. **Velocity, acceleration, jerk** per instrument per trial.
5. **3D trajectories** as colored lines (color = normalized time) per trial.
6. **Summative figures**: average force per trial and average velocity /
   acceleration / jerk per instrument.

> Runs as-is in **Google Colab**. Upload your `.igs.mha` files when prompted, or
> mount Google Drive and point `DATA_DIR` at the folder that holds them.

## 1 · Setup

In [ ]:
# Colab already ships numpy / scipy / matplotlib; this is a no-op there and a
# convenience when running elsewhere.
import importlib, subprocess, sys
for pkg in ("numpy", "scipy", "matplotlib"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, re, glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection

# ---- Plot style ---------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelsize": 10, "legend.fontsize": 9, "legend.frameon": False,
    "font.size": 10,
})

INSTRUMENTS = ["Bipolar", "Cavitron", "Scissors"]
# Okabe-Ito colorblind-safe palette, fixed order per instrument.
INST_COLOR = {"Bipolar": "#0072B2", "Cavitron": "#E69F00", "Scissors": "#009E73"}
# Sequential map used everywhere "value = time".
TIME_CMAP = "viridis"
print("Setup complete.")

## 2 · Locate the data

Put every trial's `.igs.mha` file in a folder and set `DATA_DIR` to it.

- **Colab, quick upload** — leave `DATA_DIR = "data"` and run the upload cell.
- **Colab + Google Drive** — mount Drive and set `DATA_DIR` to your folder there.
- **Local Jupyter** — set `DATA_DIR` to the repo's `data/` folder.

In [ ]:
DATA_DIR = "data"          # folder containing the *.igs.mha files
TIME_UNIT = "s"            # timestamps are in seconds
SMOOTH_WINDOW = 11         # Savitzky-Golay window (odd, in frames) for differentiation; set 0 to disable

os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
# --- Optional: upload files directly in Colab -----------------------------
# Uncomment in Colab to upload one or more *.igs.mha files into DATA_DIR.
#
# from google.colab import files
# uploaded = files.upload()
# for name, content in uploaded.items():
#     with open(os.path.join(DATA_DIR, name), "wb") as f:
#         f.write(content)
#
# --- Optional: mount Google Drive instead ---------------------------------
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/your_folder"

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.igs.mha")))
assert files, (f"No .igs.mha files found in {DATA_DIR!r}. "
               "Upload them (see the cell above) or fix DATA_DIR.")
print(f"Found {len(files)} trial(s):")
for f in files:
    print("  ", os.path.basename(f))

## 3 · Parse the sequence files

The `.igs.mha` header is a flat list of `Key = Value` lines. Per-frame fields are
named `Seq_Frame<idx>_<Field>`. We only read tracking, force, timestamps and the
fiducial points — the referenced video is ignored.

In [ ]:
def parse_mha(path):
    header, frames = {}, {}
    with open(path, "r", errors="ignore") as fh:
        for line in fh:
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            k, v = k.strip(), v.strip()
            m = re.match(r"Seq_Frame(\d+)_(.+)", k)
            if m:
                frames.setdefault(int(m.group(1)), {})[m.group(2)] = v
            else:
                header[k] = v
            if k == "ElementDataFile":   # end of header; pixel data (if any) follows
                break

    idx = sorted(frames)

    def as_mat(fr, key):
        return np.array([float(x) for x in fr[key].split()], float).reshape(4, 4)

    # 4 fiducial points collected with the Bipolar tip
    n_pts = int(header.get("BipolarCollectedPointCount", 0))
    fiducials = (np.array([[float(x) for x in header[f"BipolarCollectedPoint{i}"].split()]
                           for i in range(n_pts)]) if n_pts else None)

    timestamps = np.array([float(frames[i]["Timestamp"]) for i in idx])
    # Force = fx fy fz tx ty tz
    force = np.array([[float(x) for x in frames[i]["Force"].split()] for i in idx])

    poses, status = {}, {}
    for inst in INSTRUMENTS:
        key = inst + "TipToWorldTransform"
        if key in frames[idx[0]]:
            poses[inst] = np.stack([as_mat(frames[i], key) for i in idx])          # (N,4,4)
            status[inst] = np.array([frames[i].get(key + "Status", "OK") == "OK"
                                     for i in idx])

    return dict(name=os.path.basename(path).replace(".igs.mha", ""),
                header=header, timestamps=timestamps, force=force,
                poses=poses, status=status, fiducials=fiducials)


trials = [parse_mha(f) for f in files]
for tr in trials:
    dur = tr["timestamps"][-1] - tr["timestamps"][0]
    print(f"{tr['name']:<40s}  {len(tr['timestamps']):5d} frames  "
          f"{dur:6.1f} {TIME_UNIT}  instruments={list(tr['poses'])}")

## 4 · Point-wise registration to a common reference frame

Each trial carries the **same 4 physical fiducials** (`BipolarCollectedPoint0..3`).
We compute the rigid transform (rotation + translation, no scaling) that best maps
each trial's fiducials onto a single **reference set** — by default the first
trial's points — using the closed-form SVD solution (Kabsch / Horn's absolute
orientation). Applying that transform to every `*TipToWorldTransform` position
places all trials in one shared frame, so trajectories can be overlaid and compared.

The per-trial fiducial **RMSE** below reports how well the 4 points align; a large
value flags a mislabeled or mis-collected fiducial.

In [ ]:
def rigid_register(src, dst):
    "Least-squares rigid transform (4x4) mapping src points onto dst; returns (G, rmse)."
    src, dst = np.asarray(src, float), np.asarray(dst, float)
    cs, cd = src.mean(0), dst.mean(0)
    H = (src - cs).T @ (dst - cd)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))          # guard against reflection
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    t = cd - R @ cs
    G = np.eye(4); G[:3, :3] = R; G[:3, 3] = t
    rmse = np.sqrt(np.mean(np.sum(((R @ src.T).T + t - dst) ** 2, axis=1)))
    return G, rmse


REFERENCE_TRIAL = 0                       # which trial's fiducials define the frame
ref_pts = trials[REFERENCE_TRIAL]["fiducials"]

for tr in trials:
    if tr["fiducials"] is not None and ref_pts is not None:
        tr["G"], tr["rmse"] = rigid_register(tr["fiducials"], ref_pts)
    else:
        tr["G"], tr["rmse"] = np.eye(4), np.nan

    # Registered tip positions for every instrument.
    tr["pos"] = {}
    for inst, P in tr["poses"].items():
        xyz = P[:, :3, 3]
        tr["pos"][inst] = (tr["G"][:3, :3] @ xyz.T).T + tr["G"][:3, 3]

    # Time normalized to [0, 1].
    t = tr["timestamps"]
    tr["tnorm"] = (t - t[0]) / (t[-1] - t[0])

    print(f"{tr['name']:<40s}  registration RMSE = {tr['rmse']:.3f} mm")

## 5 · Force magnitude and motion derivatives

- **Force magnitude** `|F| = √(fx² + fy² + fz²)` (magnitude is invariant to registration).
- **Velocity, acceleration, jerk** are the 1st/2nd/3rd time-derivatives of tip
  position, magnitudes reported. Position is lightly Savitzky-Golay smoothed before
  differentiating (numerical differentiation amplifies tracking noise), and
  derivatives use the **actual timestamps**, so units are mm/s, mm/s², mm/s³.

In [ ]:
def _smooth(x, win):
    if not win or win < 3 or win >= len(x):
        return x
    if win % 2 == 0:
        win += 1
    try:
        from scipy.signal import savgol_filter
        return savgol_filter(x, win, 3, axis=0)
    except Exception:                     # fallback: centered moving average
        k = np.ones(win) / win
        return np.stack([np.convolve(x[:, j], k, mode="same") for j in range(x.shape[1])], 1)


def kinematics(pos, t, win):
    "Return speed, acceleration, jerk magnitudes from a position track."
    pos = _smooth(pos, win)
    vel = np.gradient(pos, t, axis=0)
    acc = np.gradient(vel, t, axis=0)
    jrk = np.gradient(acc, t, axis=0)
    return {"velocity": np.linalg.norm(vel, axis=1),
            "acceleration": np.linalg.norm(acc, axis=1),
            "jerk": np.linalg.norm(jrk, axis=1)}


for tr in trials:
    tr["fmag"] = np.linalg.norm(tr["force"][:, :3], axis=1)          # |force|
    tr["tmag"] = np.linalg.norm(tr["force"][:, 3:], axis=1)          # |torque| (kept for reference)
    tr["kin"] = {inst: kinematics(tr["pos"][inst], tr["timestamps"], SMOOTH_WINDOW)
                 for inst in tr["pos"]}

KIN_UNITS = {"velocity": "mm/s", "acceleration": "mm/s²", "jerk": "mm/s³"}
print("Derived force magnitude and velocity/acceleration/jerk for all trials.")

## 6 · Force magnitude per trial

In [ ]:
n = len(trials)
ncol = min(3, n); nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.2 * nrow),
                         squeeze=False, sharex=True)
for ax, tr in zip(axes.flat, trials):
    ax.plot(tr["tnorm"], tr["fmag"], color="#0072B2", lw=1.0)
    ax.fill_between(tr["tnorm"], tr["fmag"], color="#0072B2", alpha=0.12)
    ax.set_title(tr["name"], fontsize=9)
    ax.set_xlabel("normalized time"); ax.set_ylabel("|force|  (N)")
    ax.margins(x=0)
for ax in axes.flat[n:]:
    ax.set_visible(False)
fig.suptitle("Force magnitude  √(fx²+fy²+fz²)  per trial", fontweight="bold")
fig.tight_layout()
plt.show()

## 7 · Velocity, acceleration and jerk — per instrument, per trial

Rows are the three motion metrics; columns are the instruments. Each line is one
trial, plotted against normalized time.

In [ ]:
metrics = ["velocity", "acceleration", "jerk"]
fig, axes = plt.subplots(len(metrics), len(INSTRUMENTS),
                         figsize=(4.6 * len(INSTRUMENTS), 3.0 * len(metrics)),
                         squeeze=False, sharex=True)
# color trials along a light->dark ramp so overlaid lines stay distinguishable
tcolors = plt.cm.cividis(np.linspace(0.15, 0.85, len(trials)))
for r, metric in enumerate(metrics):
    for c, inst in enumerate(INSTRUMENTS):
        ax = axes[r][c]
        for tr, col in zip(trials, tcolors):
            if inst in tr["kin"]:
                ax.plot(tr["tnorm"], tr["kin"][inst][metric], lw=0.9,
                        color=col, alpha=0.9,
                        label=tr["name"][:16] if r == 0 and c == 0 else None)
        if r == 0:
            ax.set_title(inst, color=INST_COLOR[inst])
        if c == 0:
            ax.set_ylabel(f"{metric}\n({KIN_UNITS[metric]})")
        if r == len(metrics) - 1:
            ax.set_xlabel("normalized time")
        ax.margins(x=0)
if len(trials) > 1:
    axes[0][0].legend(loc="upper right", fontsize=7)
fig.suptitle("Motion derivatives per instrument (each line = one trial)", fontweight="bold")
fig.tight_layout()
plt.show()

## 8 · 3D instrument trajectories (color = normalized time)

Each instrument's registered tip path is drawn as a 3D line colored from the start
(dark) to the end (yellow) of the recording. One figure per trial; instruments
share the registered reference frame so their relative geometry is meaningful.

In [ ]:
def _color_line3d(ax, xyz, tnorm, lw=1.6):
    pts = xyz.reshape(-1, 1, 3)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = Line3DCollection(segs, cmap=TIME_CMAP, array=tnorm[:-1], linewidth=lw)
    ax.add_collection3d(lc)
    return lc

for tr in trials:
    fig = plt.figure(figsize=(6.4 * len(tr["pos"]), 5.4))
    for i, inst in enumerate(INSTRUMENTS):
        if inst not in tr["pos"]:
            continue
        ax = fig.add_subplot(1, len(tr["pos"]), i + 1, projection="3d")
        P = tr["pos"][inst]
        lc = _color_line3d(ax, P, tr["tnorm"])
        ax.set_title(inst, color=INST_COLOR[inst])
        ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)"); ax.set_zlabel("z (mm)")
        # equal aspect box
        rng = np.ptp(P, axis=0).max() / 2 or 1
        mid = P.mean(0)
        for setlim, m in zip((ax.set_xlim, ax.set_ylim, ax.set_zlim), mid):
            setlim(m - rng, m + rng)
        ax.view_init(elev=20, azim=-60)
    cb = fig.colorbar(lc, ax=fig.axes, shrink=0.6, pad=0.02)
    cb.set_label("normalized time")
    fig.suptitle(f"Registered tip trajectories — {tr['name']}", fontweight="bold")
    plt.show()

## 9 · Summative figures

Averages across all trials. Bars show the trial-mean of each metric; error bars are
the standard deviation **across trials** (or, for a single trial, the variability
across time). Force comes from one sensor (not per-instrument), so it is summarized
**per trial**; velocity / acceleration / jerk are summarized **per instrument**.

In [ ]:
def _bar(ax, labels, means, errs, colors, ylabel, title):
    x = np.arange(len(labels))
    means, errs = np.asarray(means, float), np.asarray(errs, float)
    # magnitudes are non-negative: clip the lower whisker at zero
    yerr = np.vstack([np.minimum(errs, means), errs])
    ax.bar(x, means, yerr=yerr, color=colors, width=0.62,
           error_kw=dict(lw=1, capsize=4, ecolor="#444"))
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=0)
    ax.set_ylabel(ylabel); ax.set_title(title)
    for xi, m in zip(x, means):
        ax.text(xi, m, f"{m:.2g}", ha="center", va="bottom", fontsize=8)
    ax.margins(y=0.15)

single = len(trials) == 1
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# (a) average force magnitude per trial
labels = [tr["name"][:14] for tr in trials]
fmeans = [tr["fmag"].mean() for tr in trials]
ferrs = [tr["fmag"].std() for tr in trials]
_bar(axes[0][0], labels, fmeans, ferrs,
     ["#0072B2"] * len(trials), "|force|  (N)", "Average force magnitude per trial")
if len(trials) > 3:
    axes[0][0].tick_params(axis="x", labelrotation=30)

# (b–d) average velocity / acceleration / jerk per instrument
for ax, metric in zip([axes[0][1], axes[1][0], axes[1][1]],
                      ["velocity", "acceleration", "jerk"]):
    means, errs, labs, cols = [], [], [], []
    for inst in INSTRUMENTS:
        per_trial = [tr["kin"][inst][metric].mean() for tr in trials if inst in tr["kin"]]
        if not per_trial:
            continue
        labs.append(inst); cols.append(INST_COLOR[inst])
        means.append(np.mean(per_trial))
        errs.append(np.std(tr["kin"][inst][metric]) if single else np.std(per_trial))
    _bar(ax, labs, means, errs, cols, KIN_UNITS[metric],
         f"Average {metric} per instrument")

fig.suptitle("Summary across trials", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

---
### Notes & assumptions

- **Reference frame.** Trials are registered onto trial #`REFERENCE_TRIAL`'s
  fiducials. Change `REFERENCE_TRIAL` to pick a different anchor; all trajectories
  and summaries move with it.
- **Fiducial order matters.** Point-wise registration assumes
  `BipolarCollectedPoint0..3` correspond across trials. A high RMSE in §4 usually
  means a swapped or mis-collected point.
- **Smoothing.** `SMOOTH_WINDOW` controls the Savitzky-Golay window used before
  differentiation. Larger = smoother velocity/acceleration/jerk but more lag; set
  to `0` to differentiate the raw track.
- **Force is a single sensor**, not per-instrument, so §6 and the force panel in §9
  are per trial. Torque magnitude is also parsed (`tr["tmag"]`) if you want to add it.